In [ ]:
import os
import sys

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    # Clone repository if it hasn't been cloned yet
    if not os.path.exists('/content/convolutional_vae_quishing'):
        !git clone https://github.com/AnhJoe/convolutional_vae_quishing.git

    # Change working directory to the project root
    os.chdir('/content/convolutional_vae_quishing')

In [2]:
# For local runs. Else if running Colab, skip this.

import sys
from pathlib import Path

# Set to root for src load
root = Path.cwd()
while root != root.parent and not (root / "src").exists():
    root = root.parent

if not (root / "src").exists():
    raise FileNotFoundError("Could not find 'src' folder above current working directory.")

sys.path.insert(0, str(root))
print("Project root:", root)

Project root: c:\Users\joetn\convolutional_vae_quishing


# Load Data

In [3]:
import os
import tempfile
import zipfile
from pathlib import Path

import gdown

In [4]:
# Colab will require this run for every fresh kernel runtime
# Local runs save datasets to data/raw

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1i5dQyllOAbZAM4XvXrJ0d1MuUuJMGVeb?usp=sharing"

root = globals().get("root", Path.cwd())
RAW_DIR = root / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Written after a complete download and extraction so a partial run is never mistaken for a finished one
EXTRACT_MARKER = RAW_DIR / ".extracted"


def extract_zip(zip_path: Path, dest: Path) -> Path:
    """Extract a zip into dest, skipping files that already exist with the same size.
    The folders inside the zip (QR_All_benign, QR_All_Malicious) land directly in dest,
    so the zip's own name never appears in the path. Only if an archive has loose files
    at its root is it nested under a folder named after the zip."""
    with zipfile.ZipFile(zip_path) as zf:
        infos = [i for i in zf.infolist() if not i.filename.startswith("__MACOSX")]
        tops = sorted({i.filename.split("/")[0] for i in infos if i.filename.strip("/")})
        loose_files = any(not i.is_dir() and "/" not in i.filename for i in infos)
        target = dest / zip_path.stem if loose_files else dest
        target.mkdir(parents=True, exist_ok=True)

        new, skipped = 0, 0
        for info in infos:
            out = target / info.filename
            if info.is_dir():
                out.mkdir(parents=True, exist_ok=True)
            elif out.exists() and out.stat().st_size == info.file_size:
                skipped += 1
            else:
                zf.extract(info, target)
                new += 1
    print(f"{zip_path.name}: top-level {tops}, {new:,} files extracted, {skipped:,} already present")
    return target


if EXTRACT_MARKER.exists():
    print(f"data/raw already loaded, skipping download: {RAW_DIR}")
else:
    with tempfile.TemporaryDirectory() as tmp:
        downloaded = gdown.download_folder(url=DRIVE_FOLDER_URL, output=tmp, quiet=False, use_cookies=False)
        zips = [Path(f) for f in (downloaded or []) if str(f).lower().endswith(".zip")]
        if not zips:
            raise FileNotFoundError("No zip files were downloaded. Check the Drive folder sharing settings.")
        for z in zips:
            extract_zip(z, RAW_DIR)
    EXTRACT_MARKER.touch()

Retrieving folder contents


Processing file 1M64chFyrQwzKj-9aIieYI48E5blRWERc QR_codes_dataset.zip
Processing file 1yfS63naxkxVH_6qdwMAOtPM9l7twlwvbILJo4__ZkiM README


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1M64chFyrQwzKj-9aIieYI48E5blRWERc
From (redirected): https://drive.google.com/uc?id=1M64chFyrQwzKj-9aIieYI48E5blRWERc&confirm=t&uuid=3616f912-02c0-496d-a1cc-270ad4c4dfcb
To: C:\Users\joetn\AppData\Local\Temp\tmpq12f2hbk\QR_codes_dataset.zip
100%|██████████| 640M/640M [00:56<00:00, 11.4MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1yfS63naxkxVH_6qdwMAOtPM9l7twlwvbILJo4__ZkiM
From (redirected): https://docs.google.com/document/d/1yfS63naxkxVH_6qdwMAOtPM9l7twlwvbILJo4__ZkiM/export?format=docx
To: C:\Users\joetn\AppData\Local\Temp\tmpq12f2hbk\README.docx
9.80kB [00:00, 4.65MB/s]
Download completed


QR_codes_dataset.zip: top-level ['QR_All_Malicious', 'QR_All_benign'], 1,005,740 files extracted, 0 already present


In [ ]:
# Preview what landed in data/raw
for d in sorted(p for p in RAW_DIR.iterdir() if p.is_dir()):
    n_images = sum(1 for e in os.scandir(d / "qrs") if e.name.endswith(".png"))
    print(f"{d.relative_to(root)}  ({n_images:,} images)")

# Initial EDA

# Cleaning & Transformations

## Save Processed